# Part 4 — The Work IQ tool catalog

Parts 1–3 introduced Work IQ one idea at a time. This notebook is the **complete tool catalog** —
all **10 tools** the Work IQ MCP server exposes, in the three official categories.

> **Fewer tools, more paths.** Every tool operates on a **relative resource path** (a Microsoft
> Graph endpoint). New workloads add *paths*, not tools — the surface stays fixed at 10.

| Category | Tools | What they do |
| --- | --- | --- |
| **Copilot** | `ask`, `list_agents` | Natural-language reasoning over your work context; discover agents |
| **Schema** | `search_paths`, `get_schema` | Discover which paths exist and their exact request shape at runtime |
| **Entity** | `fetch`, `call_function`, `create_entity`, `update_entity`, `delete_entity`, `do_action` | CRUD + actions on Microsoft 365 resources |

Everything runs **as the signed-in user**, honoring their Microsoft 365 permissions and sensitivity
labels. The write tools mutate real data — always confirm with a human before they run.

Docs: [Work IQ MCP tool reference](https://learn.microsoft.com/microsoft-365/copilot/extensibility/work-iq/mcp/tool-reference)

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath(".."))
from notebooks._shared import get_user_token, call_mcp, call_tool

token = get_user_token()
def mcp_call(name, arguments=None):
    return call_tool(token, name, arguments or {})
print("Signed in - Work IQ tools ready. Token:", token[:16], "...")

## The complete catalog, straight from the server

`tools/list` is authoritative: it returns every tool with its input schema. Print each tool's name
and required arguments — the ground truth for everything below.

In [ ]:
tools = call_mcp(token, "tools/list", {})["result"]["tools"]
print(f"{len(tools)} tools:\n")
for t in tools:
    req = t.get("inputSchema", {}).get("required", [])
    print(f"{t['name']:<15} required={req}")

## Copilot tools — `ask`, `list_agents`

`ask` is the highest-leverage tool: pose a question in plain language and Work IQ (Microsoft 365
Copilot) plans retrieval across mail, chats, meetings, and files, then returns a synthesized answer
with citations. Start here — reach for the lower-level tools only when you need precise control.

In [ ]:
resp = mcp_call("ask", {"question":
    "What did my manager email me about this week, and what do they need from me?"})
print(resp["result"]["structuredContent"]["answer"])

`list_agents` lists the agents you can route `ask` to (via its optional `agentId`) — the built-in
Microsoft 365 Copilot agent is always included.

In [ ]:
res = mcp_call("list_agents", {})["result"]
agents = res.get("structuredContent") or json.loads(res["content"][0]["text"])
print(json.dumps(agents, indent=2)[:600])

## Schema tools — `search_paths`, `get_schema`

Before you read or write, **discover** what's reachable. `search_paths` lists the Graph paths (and
the operations each supports) matching a regex `filter`.

In [ ]:
# search_paths returns structuredContent.paths: a list of {path, operations}.
resp = mcp_call("search_paths", {"filter": "messages|events"})
for p in resp["result"]["structuredContent"]["paths"][:10]:
    print(f"{p['path']:<48} -> {', '.join(p['operations'])}")

`get_schema` returns the request shape for a path. Pass `operationType` — one of
`fetch | create | update` — to get the shape for what you're about to do.

In [ ]:
read_schema = mcp_call("get_schema", {"path": "/me/messages", "operationType": "fetch"})
print("FETCH shape for /me/messages:\n", read_schema["result"]["content"][0]["text"][:450])

create_schema = mcp_call("get_schema", {"path": "/me/events", "operationType": "create"})
print("\nCREATE shape for /me/events:\n", create_schema["result"]["content"][0]["text"][:450])

## Entity tools — read: `fetch`, `call_function`

`fetch` reads one or more entities by URL (`entityUrls`). Use it when you know exactly what you want,
e.g. the last five messages with just the fields you care about.

In [ ]:
msgs = mcp_call("fetch", {"entityUrls": ["/me/messages?$top=5&$select=subject,from,receivedDateTime"]})
for res in msgs["result"]["structuredContent"]["results"]:
    for m in res["data"].get("value", [])[:5]:
        print(m.get("receivedDateTime"), "-", m.get("subject"))

`call_function` invokes a read-only Microsoft Graph **function** that computes derived data —
schedules, deltas, search results — via `functionUrl`. Here: this week's calendar view.

In [ ]:
from datetime import datetime, timedelta, timezone
start = datetime.now(timezone.utc).isoformat()
end = (datetime.now(timezone.utc) + timedelta(days=7)).isoformat()
resp = mcp_call("call_function",
    {"functionUrl": f"/me/calendarView?startdatetime={start}&enddatetime={end}"})
for e in resp["result"]["structuredContent"]["data"].get("value", [])[:5]:
    print(e["start"]["dateTime"], "-", e.get("subject"))

## Entity tools — write: `create_entity`, `update_entity`, `delete_entity`, `do_action`

These four **mutate real data as the signed-in user**. Two gotchas from the docs:

- `create_entity` takes **`parentUrl`** (the collection); `update_entity` / `delete_entity` take
  **`entityUrl`** (the specific item); `do_action` takes **`actionUrl`**.
- `jsonBody` must be a **JSON-encoded string**, not a JSON object — so we `json.dumps(...)` it.

Get every payload shape from `get_schema`, and **always confirm with a human first**.

In [ ]:
# All dry-run: flip WRITE=True only after a human approves the payload.
WRITE = False

create_args = {"parentUrl": "/me/events", "jsonBody": json.dumps({
    "subject": "Q3 budget sync",
    "start": {"dateTime": "2026-08-01T15:00:00", "timeZone": "UTC"},
    "end":   {"dateTime": "2026-08-01T15:30:00", "timeZone": "UTC"},
})}
update_args = {"entityUrl": "/me/events/<event-id>",
               "jsonBody": json.dumps({"subject": "Q3 budget sync (moved)"})}
delete_args = {"entityUrl": "/me/events/<event-id>"}

for tool, args in [("create_entity", create_args),
                   ("update_entity", update_args),
                   ("delete_entity", delete_args)]:
    if WRITE:
        print(tool, "->", mcp_call(tool, args))
    else:
        print(f"[dry run] {tool}({json.dumps(args)})")

`do_action` is the action verb — send mail, reply, accept an event. Here, sending mail:
`actionUrl` + a JSON-encoded `jsonBody`.

In [ ]:
SEND = False  # flip to True to actually send
action_url = "/me/sendMail"
json_body = json.dumps({
    "Message": {
        "subject": "Re: your note",
        "body": {"contentType": "Text", "content": "<approved draft here>"},
        "toRecipients": [{"emailAddress": {"address": "manager@contoso.com"}}],
    },
    "SaveToSentItems": True,
})
if SEND:
    print(mcp_call("do_action", {"actionUrl": action_url, "jsonBody": json_body}))
else:
    print("Dry run - set SEND=True to send. do_action args:")
    print(json.dumps({"actionUrl": action_url, "jsonBody": json_body}, indent=2))

### Recap

- **Copilot:** `ask` is the front door — one question, retrieval planned across your work context;
  `list_agents` shows who you can route to.
- **Schema:** `search_paths` → `get_schema` discover reachable paths and their exact shapes.
- **Entity (read):** `fetch` (entities by URL) and `call_function` (computed Graph functions).
- **Entity (write):** `create_entity` / `update_entity` / `delete_entity` and `do_action` — real
  mutations, `jsonBody` as a JSON-encoded string, human-in-the-loop.
- The pattern that scales: **ask / read context → draft → confirm → write**.

**Next:** wrap this toolset into an agent — see the Foundry IQ + Work IQ notebook and `src/workmate-host/`.